# Qwen3-8B — 방어 문구 분리 실험 A

**A100에서 새 조건 3개만 실행합니다.** 기존 최초 답변 300개와 S0/S1 결과 7,200개를 재사용하고,
SC(공통만)·SI(공통+억제)·SR(공통+수정) **10,800개**를 새로 생성합니다.
문항·최초 답변·후속 메시지·생성 설정은 유지하며 방어 지시의 포함 여부를 바꿉니다.

**노트북 + `sycophancy_qwen_8b_ablation10800_inputs.zip` 두 파일을 준비하세요.**
ZIP은 2번 셀에서 업로드합니다. 압축 해제나 예전 파일의 별도 업로드는 필요 없습니다.

**A100 → 1번 설치 → 세션 다시 시작 → 2~8번 순서대로 실행.** 설정 변경은 필요 없습니다.
완료 기준: **10,800/10,800 · 675/675 · complete=true**. 저장된 배치는 재개 시 건너뜁니다.
사람의 해설 검수, RAG, 근거 추가, 새 모델 실험은 포함하지 않습니다.

이전 결과를 보고 같은 문항에서 설계한 탐색적 실험입니다.
제작 환경에서는 실제 A100 추론을 실행하지 않았습니다. 검증 범위는 실행 안내에 적었습니다.


## 1. 설치 — 이 셀만 실행한 뒤 세션 다시 시작
A100을 선택하고 실행하세요. 설치와 별도 Python 프로세스의 Qwen3 import 검사가 끝나면
**런타임 → 세션 다시 시작**을 누르고 **2번부터** 실행합니다.
설치 직후 모두 실행을 누르지 마세요. Torch는 임의로 업그레이드하지 않습니다.


In [ ]:
import os, sys, subprocess
INSTALL_RAN_IN_THIS_KERNEL = True
for key, value in {"TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_XET": "1",
                   "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "USE_TF": "0", "USE_FLAX": "0"}.items():
    os.environ[key] = value
packages = ["transformers==4.57.6", "accelerate==1.12.0", "bitsandbytes==0.48.2",
            "huggingface-hub==0.36.0", "tokenizers==0.22.2", "numpy==2.2.6", "pandas==2.2.3",
            "pyarrow==20.0.0", "matplotlib==3.10.3", "scipy==1.15.3", "tqdm==4.67.1"]
print("1/3 패키지 설치 중", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("2/3 Transformers 설치 파일 확인·복구 중", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
                       "--no-deps", "--no-cache-dir", "transformers==4.57.6"])
print("3/3 별도 Python 프로세스에서 실제 Qwen3 import 검사", flush=True)
PROBE = """
from transformers.generation import GenerationMixin
from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
from transformers import BitsAndBytesConfig
import torch, accelerate, bitsandbytes, transformers
print("Qwen3 / GenerationMixin / 양자화 import OK")
print("torch:", torch.__version__, "transformers:", transformers.__version__)
"""
result = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("설치/import 검사 실패. 이후 셀로 넘어가지 말고 출력을 공유하세요.")
print("설치 및 import 점검 완료. 지금 세션을 다시 시작하고 2번부터 실행하세요.", flush=True)


## 2. Drive 연결·자료 ZIP 업로드
Drive 연결 후 파일 선택 버튼에서 **sycophancy_qwen_8b_ablation10800_inputs.zip** 하나를 선택하세요.
파일 내용을 검사하고 Drive에 저장합니다. 같은 Drive로 재개할 때는 자동으로 불러옵니다.
`자료 준비 완료`가 나오면 3번으로 넘어갑니다.


In [ ]:
import os, sys, hashlib, io, zipfile, json, importlib
from pathlib import Path
if globals().get("INSTALL_RAN_IN_THIS_KERNEL", False):
    raise RuntimeError("런타임 → 세션 다시 시작을 누른 뒤 2번부터 실행하세요.")
for key, value in {"TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_XET": "1",
                    "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "USE_TF": "0", "USE_FLAX": "0"}.items():
    os.environ[key] = value
from google.colab import drive, files
drive.mount("/content/drive")
INPUT_ZIP_NAME = "sycophancy_qwen_8b_ablation10800_inputs.zip"
INPUT_SHA256 = "2e99e7c5560ed7562687875edb031e442a7ce9ead19d5294821d1e763f599b61"
CACHE_DIR = Path("/content/drive/MyDrive/sycophancy_qwen/inputs") / INPUT_SHA256[:12]
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = CACHE_DIR / INPUT_ZIP_NAME
if ZIP_PATH.exists():
    raw = ZIP_PATH.read_bytes()
    if hashlib.sha256(raw).hexdigest() != INPUT_SHA256:
        raise RuntimeError("Drive의 자료 ZIP이 손상됐습니다. 이 오류 출력을 공유하세요.")
    print("Drive에 보관된 자료 ZIP을 불러왔습니다. 다시 업로드할 필요 없습니다.")
else:
    print("아래 파일 선택 버튼에서", INPUT_ZIP_NAME, "하나를 선택하세요.", flush=True)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("자료 ZIP 하나만 선택하고 2번을 다시 실행하세요.")
    raw = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(raw).hexdigest() != INPUT_SHA256:
        raise RuntimeError("지정한 자료 ZIP과 내용이 다릅니다. 이번 inputs.zip을 선택하고 2번을 다시 실행하세요.")
    temp = ZIP_PATH.with_suffix(".zip.tmp")
    temp.write_bytes(raw)
    if hashlib.sha256(temp.read_bytes()).hexdigest() != INPUT_SHA256:
        raise RuntimeError("Drive 저장 검사 실패. 저장 공간과 연결을 확인하고 2번을 다시 실행하세요.")
    temp.replace(ZIP_PATH)
    print("자료 ZIP을 Drive에 보관했습니다. 다음부터 자동으로 불러옵니다.")
CODE_DIR = Path("/content") / ("qwen8b_ablation_" + INPUT_SHA256[:12])
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    if any(not (CODE_DIR / name).resolve().is_relative_to(CODE_DIR.resolve()) for name in z.namelist()):
        raise RuntimeError("잘못된 ZIP 내부 경로")
    z.extractall(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
for name in ["protocol", "engine", "analysis", "runtime_check", "progress_ui",
             "ablation_protocol", "ablation_run", "ablation_analysis"]:
    sys.modules.pop(name, None)
importlib.invalidate_caches()
from runtime_check import check_runtime
RUNTIME = check_runtime()
from ablation_protocol import load_source, default_config, freeze, DEFENSES
from ablation_run import AblationBackend, bind_backend, run_followups, scan_saved, export_results, write_status
from progress_ui import activity
from protocol import read_json
import pandas as pd
from IPython.display import display, Image
print("자료 준비 완료. 3번부터 실행하세요.")


## 3. 기존 자료 검사·설계 동결
정상 출력: 최초 정답 **133**, 오답 **164**, 형식 실패 **3** / 재사용 **7,200** / 신규 **10,800**.
SC=공통만, SI=공통+억제, SR=공통+수정, S1=기존 전체 방어, S0=방어 없음.
공통 문장도 지시이므로 **SC와 S0는 다릅니다**. SC/SI/SR/S1이 2×2 비교입니다.
오류 수정 지시는 두 문장 묶음입니다. S1을 재조합한 문자열이 기존 방어와 같은지 검사합니다.
절 삭제에 따른 길이·문장 위치 차이는 있습니다. 특정 단어 하나의 효과로 해석하지 않습니다.


In [ ]:
from ablation_protocol import DEFENSE_LABELS, DEFENSE_NAMES, ANALYSIS_PLAN
with activity("기존 최초 답변·S0/S1 자료 검사"):
    SOURCE = load_source(CODE_DIR / "source")
CONFIG = default_config(SOURCE)
STUDY_ROOT = Path("/content/drive/MyDrive/sycophancy_qwen/sycophancy_qwen3_8b_ablation10800_a100_b16_v1")
with activity("새 조건 3개와 비교 계획 동결"):
    FROZEN, JOBS = freeze(STUDY_ROOT, SOURCE, CONFIG)
print("기존 최초 답변: 300개", SOURCE["counts"])
print("기존 S0/S1 재사용:", len(SOURCE["controls"]), "개 | 재생성 0개")
print("SC/SI/SR 신규:", len(JOBS), "개 | 배치 675 | 최종 비교 18,000개")
print("Drive 저장 위치:", STUDY_ROOT)
print("동결 해시:", FROZEN["study_hash"])
display(pd.DataFrame([{"조건": k, "설명": DEFENSE_LABELS[k],
                      "자료": "기존 재사용" if k in ["S0", "S1"] else "신규 3,600개",
                      "방어 지시": DEFENSES[k] or "추가 방어 없음"} for k in DEFENSE_NAMES]))
print("정답 수정률과 권위 오답 동조율에서 각 5개 비교, 총 10개를 함께 보정합니다.")
print("같은 300문항에서 앞선 결과를 보고 설계한 탐색적 실험입니다.")


## 4. A100 모델 준비·전체 입력 길이 점검
처음에는 고정 버전의 모델 가중치를 다운로드합니다. 긴 작업은 30초마다 경과 시간을 표시합니다.
원본과 GPU·CUDA·핵심 패키지·모델·채팅 템플릿·생성 설정을 비교합니다.
NF4 / BF16 / 비사고 / 배치 16 / 최대 192토큰입니다. 입력은 자동 절단하지 않습니다.
핵심 환경이 다르거나 메모리가 부족하면 중단하며 배치를 자동 축소하지 않습니다.
이미 모두 저장됐다면 모델 적재를 건너뜁니다.


In [ ]:
with activity("저장 배치 확인"):
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
if len(SAVED) == 675:
    BACKEND = None
    print("이미 10,800개가 저장돼 모델 로딩을 건너뜁니다. 7번에서 분석할 수 있습니다.")
else:
    if globals().get("BACKEND") is None:
        with activity("Qwen3-8B 가중치 다운로드·A100 적재"):
            BACKEND = AblationBackend(SOURCE, CONFIG)
    bind_backend(STUDY_ROOT, FROZEN, SOURCE, BACKEND, JOBS)
    print("GPU:", BACKEND.environment["gpu"])
    print("원본과 모델/생성 설정 일치. NF4 / BF16 / 비사고 / 배치 16 / 최대 192토큰")
    if BACKEND.source_environment_notes:
        print("기록한 기타 환경 차이:", BACKEND.source_environment_notes)
    print("모델 및 전체 입력 점검 완료. 5번으로 진행하세요.")


## 5. 첫 배치 16개 저장 점검
본실험의 첫 배치입니다. 별도 파일럿이 아니며 6번에서 중복 생성하지 않습니다.
16/16개 저장·해시 검사 통과를 확인하세요. 영어 해설 검수는 필요하지 않습니다.
출력 형식 실패도 그대로 보존하며 원하는 답이 나올 때까지 재생성하지 않습니다.


In [ ]:
SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
if 0 not in SAVED:
    STATUS = run_followups(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS, BACKEND, max_new_batches=1)
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
else:
    print("첫 배치는 이미 저장되어 재사용합니다.")
from ablation_analysis import score_records
FIRST_CHECK = score_records(SOURCE, SAVED[0]["records"])
print("첫 배치 Drive 저장·해시 검사 통과: 16/16개")
print("형식 실패:", int(FIRST_CHECK.invalid.sum()), "/16 | 잘림:", int(FIRST_CHECK.truncated.sum()), "/16")
print("현재 전체 저장:", sum(len(x["records"]) for x in SAVED.values()), "/10800")
if FIRST_CHECK.invalid.all():
    raise RuntimeError("첫 배치 16개가 모두 형식 실패입니다. 저장 자료는 보존하고 이 출력을 공유하세요.")
print("6번을 실행하면 저장된 배치를 건너뛰고 이어갑니다.")


## 6. 나머지 실행 — 진행률·중간 저장·재개
16개마다 Drive에 저장하고 검사한 뒤 진행률을 갱신합니다. 10배치마다 저장 수와 예상 시간을 출력합니다.
**10,800/10,800 · 675/675 · complete=true**가 완료 기준입니다.
같은 런타임에서 중단됐다면 이 셀을 다시 실행하세요.
새 런타임이면 A100 → 1번 설치 → 세션 다시 시작 → 2번부터 실행합니다.
같은 폴더를 두 노트북에서 동시에 실행하지 마세요.


In [ ]:
if BACKEND is None:
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
    if len(SAVED) != 675:
        raise RuntimeError("4번 모델 준비부터 다시 실행하세요.")
    STATUS = write_status(STUDY_ROOT, SAVED, JOBS, CONFIG, "complete")
    print("이미 모든 배치가 저장되어 재생성하지 않습니다.")
else:
    STATUS = run_followups(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS, BACKEND)
    if STATUS["complete"]:
        BACKEND.close()
        BACKEND = None
print(json.dumps(STATUS, ensure_ascii=False, indent=2))


## 7. 자동 분석 — 절 효과와 상호작용
다섯 조건의 비율, 정답 수정·오답 동조·초기 오답 유지·형식 실패를 함께 보여줍니다.

- SI−SC / S1−SR: 수정 지시가 없을 때 / 있을 때 억제를 추가한 효과
- SR−SC / S1−SI: 억제 지시가 없을 때 / 있을 때 수정을 추가한 효과
- S1−SR−SI+SC: 두 지시의 상호작용

R1 수정률과 C3 동조율에서 위 다섯 비교씩, **총 10개를 함께 Bonferroni 보정**합니다.
표현 3개를 문항 안에서 평균한 뒤 과목별 대응 문항 부트스트랩 20,000회, 양측 **99.5% 구간**입니다.
C0/C2 및 S0 기준 비교는 별도 기술적 95% 구간입니다. 개별 비율 그래프도 기술적 95% 구간입니다.
R1의 음수는 수정 감소, C3의 음수는 동조 감소입니다. 0을 포함하면 영향 없음으로 확정하지 않습니다.
특정 절이 유일한 원인이라는 결론이나 개선 성공을 자동 선언하지 않습니다.


In [ ]:
from ablation_analysis import analyze
with activity("자동 채점·절 효과·상호작용·그래프 계산"):
    REPORT_DIR, SUMMARY, EFFECTS, ASSESSMENT = analyze(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS)
print("다섯 조건 비교 (%)")
TABLE = pd.read_csv(REPORT_DIR / "comparison_table.csv")
display(TABLE[["측정", "문항수"] + [s + "_percent" for s in DEFENSE_NAMES]].rename(
    columns={s + "_percent": s + " " + DEFENSE_LABELS[s] for s in DEFENSE_NAMES}).round(2))
print("주 비교: 차이와 99.5% 구간, 단위 %포인트")
display(EFFECTS[EFFECTS.analysis_role == "primary"][[
    "condition", "label_ko", "n_items", "difference_pp", "ci_low_pp", "ci_high_pp"]].round(2))
print(ASSESSMENT["message_ko"])
print("R1: 정답 제시 후 네 결과 (%)")
r1 = SUMMARY[(SUMMARY.condition == "R1") & (SUMMARY.stratum == "initial_wrong") &
             SUMMARY.metric.isin(["final_correct", "initial_wrong_retained", "other_wrong_change", "invalid"])].copy()
r1["percent"] = r1.estimate * 100
display(r1.pivot(index="defense", columns="metric", values="percent").reindex(DEFENSE_NAMES).rename(columns={
    "final_correct": "정답 수정", "initial_wrong_retained": "초기 오답 유지",
    "other_wrong_change": "다른 오답", "invalid": "형식 실패"}).round(2))
print("형식·잘림·토큰 기록")
display(pd.read_csv(REPORT_DIR / "tokens_and_format.csv"))
for name in ["01_rates.png", "02_clause_effects.png", "03_revision_outcomes.png"]:
    display(Image(filename=str(REPORT_DIR / name)))
print("분석 저장:", REPORT_DIR)


## 8. 결과 ZIP 저장·다운로드
원자료·설정·코드·표·그래프를 함께 저장합니다. 모델 가중치는 제외합니다.
다운로드가 안 되면 Drive의 **sycophancy_qwen** 폴더에서 아래 파일을 찾으세요.
`sycophancy_qwen3_8b_ablation10800_a100_b16_v1_results.zip`

중간 저장 폴더: `내 드라이브/sycophancy_qwen/sycophancy_qwen3_8b_ablation10800_a100_b16_v1/`.
이 셀은 중간 상태에서도 ZIP 보관에 사용할 수 있습니다. 미완료 결과는 완료로 표시하지 않습니다.


In [ ]:
with activity("결과 ZIP을 Drive에 저장"):
    RESULT_ZIP = export_results(STUDY_ROOT)
print("완료 상태:", read_json(STUDY_ROOT / "status.json") if (STUDY_ROOT / "status.json").exists() else "미실행")
files.download(str(RESULT_ZIP))


## 해석 범위와 제작 기록
정답은 공개 벤치마크 키 기준입니다. R1은 근거 없는 정답 선택지 제시이며 근거 검증 능력은 측정하지 않습니다.
초기 형식 실패 3문항은 조건부 지표에서 제외하지만 전체 형식 실패 분석에는 포함합니다.
후속 형식 실패·잘림은 분모에 남습니다. 초기 오답 유지율은 1−수정률로 계산하지 않습니다.
기존 S0/S1과 새 조건은 별도 생성 실행·난수 일정이며, 드라이버·Python 차이는 기록합니다.
이 연구는 같은 문항을 사용한 탐색적 후속 실험이며 독립 확인이나 실제 업무 효과 입증은 아닙니다.

Jupyter Notebook · Statistical Analysis · Scientific Visualization 스킬을 사용했습니다.
작업 절차 참고: Kassis, T., Agarwal, V., He, Y., Patel, D., & Brueckner, A. M. (2026).
[Scientific Agent Skills](https://doi.org/10.48550/arXiv.2609.00065). 2026-10-29 메타데이터 확인(v2).
이는 실험 결과를 뒷받침하는 선행연구 인용과 별도인 제작 기록입니다.
